# Fase 1 — Planejamento do objetivo e do modelo

**Projeto:** Fine-Tuning para Triagem de Alertas em SOC
**Ambiente:** Google Colab (GPU T4) + Hugging Face Hub

Notebook de demonstração da Fase 1. Nesta fase vamos:

1. Definir a tarefa do modelo e as metas de sucesso.
2. Escolher o modelo base.
3. Rodar o **teste de necessidade**: medir se um prompt few-shot no modelo base já resolve a triagem. Se resolver, não há por que fazer fine-tuning.

Cada passo traz três partes:

- **O que fazer:** a tarefa do passo.
- **Por que:** o conceito por trás dela.
- **Solução:** a resposta ou o código.

## Política de severidade da organização

É o que o modelo deve aprender. A severidade **não** depende só do sinal técnico. Depende da combinação com o contexto. Exemplo resumido:

| Sinal técnico | Ativo crítico (DC, servidor financeiro) | Ativo comum | Ativo de laboratório |
|---|---|---|---|
| Forte (ex.: PowerShell codificado filho do Outlook, IOC com reputação maliciosa) | crítica | alta | media |
| Moderado (ex.: login fora do horário, ferramenta administrativa incomum) | alta | media | baixa |
| Fraco e recorrente (ex.: mesmo alerta > 50 vezes em 7 dias sem incidente) | media | baixa | baixa (falso positivo provável) |

As metas de sucesso estão no Passo 4.

## Antes de começar

1. **Conta no Hugging Face:** crie uma conta em [huggingface.co](https://huggingface.co) e aceite o convite para a *organization* da turma. O dataset do projeto é privado e só pode ser lido por membros dela.
2. **Token de acesso:** em *Settings → Access Tokens*, crie um token do tipo **Read**. Nas fases de publicação é preciso um token **Write**.
3. **Abrir o notebook no Colab:** use o link "Abrir no Colab" da tabela do README e, em seguida, *Arquivo → Salvar uma cópia no Drive*. Sem a cópia, as alterações se perdem.
4. **GPU:** *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*.
5. **Secrets:** no ícone de chave do Colab, crie o segredo `HF_TOKEN` com o token e libere o acesso do notebook.

## Regras de segurança

- O token do Hugging Face fica **só** nos *Secrets* do Colab. Nunca cole o token numa célula, num arquivo ou num commit.
- Modelos e datasets publicados no Hugging Face devem ser **privados**.
- Todos os dados do projeto são **sintéticos**. Nenhum dado real de incidente deve ser usado.

## Passo 1 — Preparar o ambiente no Colab

**O que fazer:** ativar a GPU, instalar as bibliotecas e autenticar no Hugging Face.

**Por que:** o runtime do Colab começa vazio a cada sessão. Sem GPU, o modelo roda na CPU e o teste da Fase 1 fica lento demais. O login é necessário para ler o dataset privado da turma no Hub.

**Solução:** com a GPU T4 selecionada e o segredo `HF_TOKEN` criado (ver *Antes de começar*), rodar a célula de setup.

> **Erro comum:** colar o token direto na célula. Se o notebook for salvo no GitHub, o token vaza. O token só fica nos *Secrets*.

In [ ]:
!pip install -q transformers datasets peft trl evaluate scikit-learn

from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

Confirmar que a GPU está ativa:

In [ ]:
import torch

print(torch.cuda.is_available(), torch.cuda.get_device_name(0))
# Esperado: True Tesla T4

## Passo 2 — Definir a tarefa

**O que fazer:** escrever em uma frase o que o modelo recebe, o que devolve e qual tipo de fine-tuning será usado.

**Por que:** sem uma tarefa fechada, não há como definir métrica, montar o dataset nem escolher o modelo. A tarefa também delimita o que **não** é papel do modelo: enriquecimento, mapeamento MITRE e resposta ficam em outras etapas do SOC.

**Solução:**

| Item | Definição |
|---|---|
| **Entrada** | Alerta com sinal técnico (fonte, host, usuário, processo) + contexto já enriquecido (criticidade do ativo, perfil do usuário, reputação de IOC, ocorrências em 7 dias) |
| **Saída** | JSON com exatamente 3 chaves: `veredito`, `severidade` e `justificativa` |
| **Tipo** | Fine-tuning supervisionado (SFT) de um LLM pequeno instruction-tuned, com saída em formato restrito |
| **Fora do escopo** | Consultar bases externas, mapear MITRE ATT&CK, decidir a ação de resposta |

Formato da saída:

```json
{
  "veredito": "falso_positivo | verdadeiro_positivo",
  "severidade": "baixa | media | alta | critica",
  "justificativa": "explicação curta citando o sinal técnico e o contexto"
}
```

> **Por que um LLM e não um classificador simples?** Veredito e severidade poderiam vir de um classificador, mas a `justificativa` em linguagem natural exige um modelo gerativo. É ela que permite ao analista L1 validar a triagem rapidamente.

## Passo 3 — Aplicar a política de severidade à mão

**O que fazer:** usando a tabela da política de severidade (início do notebook), triar manualmente os 4 alertas abaixo.

**Por que:** antes de ensinar uma regra ao modelo, a equipe precisa saber aplicá-la. O exercício mostra que a severidade depende da **combinação** de sinal técnico com contexto, e não só do sinal. É exatamente isso que o fine-tuning deve capturar.

Alertas:

- **A)** EDR: `powershell.exe -enc ...` com processo pai `winword.exe` no host `DC-01` (controlador de domínio). Ocorrências em 7 dias: 1.
- **B)** EDR: execução de `psexec.exe` pelo usuário `m.souza` (analista de RH, sem privilégio admin) na estação `WS-RH-14`. Ocorrências em 7 dias: 1.
- **C)** IDS: varredura de portas a partir do scanner interno `LAB-SCAN-01` no ativo `LAB-WEB-03`. Ocorrências em 7 dias: 120, nenhum incidente associado.
- **D)** EDR: hash com reputação **maliciosa** na Threat Intel executado na máquina `LAB-TEST-07` (laboratório). Ocorrências em 7 dias: 1.

**Solução:**

| Alerta | Sinal técnico | Ativo | Veredito | Severidade | Justificativa |
|---|---|---|---|---|---|
| A | Forte (PowerShell codificado disparado pelo Word) | Crítico | verdadeiro_positivo | **critica** | PowerShell codificado iniciado pelo Word num controlador de domínio é padrão típico de macro maliciosa no ativo mais sensível da rede. |
| B | Moderado (ferramenta administrativa incomum para o perfil) | Comum | verdadeiro_positivo | **media** | PsExec não é esperado para um usuário de RH sem privilégio admin. Pede verificação, mas a estação não é crítica. |
| C | Fraco e recorrente | Laboratório | **falso_positivo** | **baixa** | Varredura recorrente vinda do scanner interno, 120 vezes em 7 dias sem incidente. É atividade autorizada de laboratório. |
| D | Forte (IOC com reputação maliciosa) | Laboratório | verdadeiro_positivo | **media** | O hash é malicioso, mas a máquina é de laboratório e isolada do ambiente produtivo. O sinal forte mantém o veredito positivo. |

> **Ponto-chave:** A e D têm sinal técnico **forte**, mas severidades diferentes (crítica × media) por causa do ativo. Um modelo genérico tende a dar a mesma severidade para os dois. O modelo ajustado precisa aprender essa diferença.

## Passo 4 — Definir os critérios de sucesso

**O que fazer:** fixar as metas numéricas **antes** de treinar e justificar cada uma.

**Por que:** definir a meta depois de ver o resultado é "mover a trave". Metas fixadas antes deixam a avaliação da Fase 6 honesta. Em segurança, os erros não custam igual, então a métrica principal não pode ser só a acurácia.

**Solução:**

| Métrica | Meta | Por que essa métrica |
|---|---|---|
| JSON válido com as 3 chaves | ≥ 98% | A triagem é consumida por sistemas (fila do SOC, SOAR). Uma saída que não é JSON válido trava o fluxo. |
| Acurácia de veredito | ≥ 85% | Mede se o modelo separa bem falso positivo de verdadeiro positivo, que é o maior ganho de tempo para o L1. |
| **Recall de severidade "crítica"** | **≥ 90%** | **Métrica de negócio.** De todos os alertas realmente críticos, quantos o modelo marcou como críticos. Um crítico classificado como "media" vai para o fim da fila: é um ataque real ignorado. |
| Justificativa coerente (checklist humano, 10 casos) | ≥ 8 de 10 | O analista só confia na triagem se a justificativa cita o sinal técnico e o contexto e é coerente com a severidade. |
| Ganho sobre o baseline | Superar o baseline | Se o prompt sozinho já resolve, não há por que fazer fine-tuning. |

> **Por que recall e não precisão para "crítica"?** Uma precisão baixa significa marcar como crítico algo que não é: custa alguns minutos do analista. Um recall baixo significa deixar de marcar um crítico real: pode custar um incidente. Em SOC, o segundo erro é muito mais caro.

## Passo 5 — Escolher o modelo base

**O que fazer:** comparar modelos candidatos e escolher um, conferindo a licença e o acesso no Hub.

**Por que:** o modelo base define o que é possível nas fases seguintes. Se ele não tem chat template, a Fase 3 muda. Se é grande demais, o treino não cabe na T4. Se a licença é restritiva, a Fase 7 fica bloqueada. E o modelo precisa ser pequeno o bastante para rodar on-premises em volume, que é a justificativa do projeto.

**Solução:** primeiro, conferir licença, acesso e tamanho direto pelo Hub.

In [ ]:
from huggingface_hub import model_info

candidatos = [
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "meta-llama/Llama-3.2-1B-Instruct",
    "microsoft/Phi-3.5-mini-instruct",
]

for model_id in candidatos:
    info = model_info(model_id)
    licenca = info.card_data.license if info.card_data else None
    params = info.safetensors.total if info.safetensors else None
    print(f"{model_id:40} licença={licenca} gated={info.gated} params={params}")

Comparação:

| Critério | Qwen2.5-0.5B-Instruct | Qwen2.5-1.5B-Instruct | Llama-3.2-1B-Instruct | Phi-3.5-mini-instruct |
|---|---|---|---|---|
| Instruction-tuned (chat template) | Sim | Sim | Sim | Sim |
| Licença | Apache 2.0 | Apache 2.0 | Llama 3.2 Community | MIT |
| Acesso *gated* | Não | Não | **Sim** (exige aprovação) | Não |
| Treina numa T4 no tempo do projeto | Sim (~15 min com LoRA) | Sim, mais lento | Sim | Apertado (3,8B parâmetros) |
| Português | Bom | Bom | Razoável | Razoável |
| Implantação on-premises em volume | Excelente | Boa | Boa | Mais cara |

Decisão:

- **Escolhido: `Qwen/Qwen2.5-0.5B-Instruct`** ([página no Hub](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)). Licença Apache 2.0, sem *gating* (ninguém da turma fica bloqueado esperando aprovação), bom em português, treina rápido numa T4 e é barato de rodar em volume.
- **Alternativa:** `Qwen/Qwen2.5-1.5B-Instruct`, se sobrar tempo ou se a justificativa do 0.5B ficar fraca na Fase 6.
- **Descartados:** Llama 3.2 pelo acesso *gated*, que cria risco de bloqueio no meio do projeto, e Phi-3.5-mini pelo tamanho, que compromete o tempo de treino.

## Passo 6 — Carregar o modelo e conferir o chat template

**O que fazer:** carregar o modelo escolhido na GPU, ver como o chat template formata uma conversa e medir a memória usada.

**Por que:** modelos instruction-tuned esperam um formato de conversa específico (system/user/assistant), com tokens especiais próprios. Ver esse formato agora prepara a Fase 3. A memória confirma que sobra espaço na T4 para o treino.

**Solução:**

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")

exemplo = [
    {"role": "system", "content": "Você é um analista de SOC."},
    {"role": "user", "content": "Alerta de teste"},
]
print(tokenizer.apply_chat_template(exemplo, tokenize=False, add_generation_prompt=True))
print(f"Memória da GPU: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Saída esperada (formato do Qwen):

```
<|im_start|>system
Você é um analista de SOC.<|im_end|>
<|im_start|>user
Alerta de teste<|im_end|>
<|im_start|>assistant
```

O modelo 0.5B em `fp16` ocupa cerca de 1 GB dos 16 GB da T4.

> `add_generation_prompt=True` acrescenta o `<|im_start|>assistant` no final, que sinaliza ao modelo que é a vez dele responder. Sem isso, a geração pode continuar o texto do usuário.

## Passo 7 — Montar o prompt few-shot do baseline

**O que fazer:** escrever um prompt com a política de severidade e 3 exemplos resolvidos, que será o baseline do projeto.

**Por que:** o teste de necessidade só é justo se o baseline for **o melhor prompt razoável**, não um prompt fraco feito para perder. Few-shot (alguns exemplos resolvidos dentro do prompt) é a alternativa mais barata ao fine-tuning e é a técnica que se tentaria primeiro no mundo real.

**Solução:**

> **Cuidado com vazamento:** os 3 exemplos do few-shot foram escritos à mão e **não** vêm da amostra de baseline. Usar alertas da própria amostra como exemplo inflaria o resultado.

In [ ]:
import json

POLITICA = """Política de severidade da organização:
- Sinal forte: ativo crítico = critica; ativo comum = alta; ativo de laboratório = media.
- Sinal moderado: ativo crítico = alta; ativo comum = media; ativo de laboratório = baixa.
- Sinal fraco e recorrente: ativo crítico = media; ativo comum = baixa; ativo de laboratório = baixa (falso positivo provável).
Responda APENAS com um JSON com as chaves: veredito (falso_positivo | verdadeiro_positivo), severidade (baixa | media | alta | critica) e justificativa."""

SYSTEM_BASELINE = f"Você é um analista de SOC.\n{POLITICA}"

FEW_SHOT = [
    (
        "EDR | Host: DC-01 | Usuário: svc_backup | Processo: powershell.exe -enc JAB... | Pai: winword.exe\n"
        "Ativo: controlador de domínio (criticidade alta) | Usuário: conta de serviço | Reputação: desconhecida | Ocorrências em 7 dias: 1",
        {"veredito": "verdadeiro_positivo", "severidade": "critica",
         "justificativa": "PowerShell codificado iniciado pelo Word em controlador de domínio, padrão de macro maliciosa no ativo mais sensível."},
    ),
    (
        "EDR | Host: WS-RH-14 | Usuário: m.souza | Processo: psexec.exe\n"
        "Ativo: estação comum | Usuário: RH, sem privilégio admin | Reputação: legítima | Ocorrências em 7 dias: 1",
        {"veredito": "verdadeiro_positivo", "severidade": "media",
         "justificativa": "Ferramenta administrativa incomum para o perfil do usuário, em estação não crítica."},
    ),
    (
        "IDS | Origem: LAB-SCAN-01 | Destino: LAB-WEB-03 | Evento: varredura de portas\n"
        "Ativo: laboratório | Origem: scanner interno autorizado | Ocorrências em 7 dias: 120, sem incidente",
        {"veredito": "falso_positivo", "severidade": "baixa",
         "justificativa": "Varredura recorrente do scanner interno autorizado em ambiente de laboratório."},
    ),
]


def montar_mensagens(alerta: str, contexto: str) -> list[dict]:
    messages = [{"role": "system", "content": SYSTEM_BASELINE}]
    for entrada, saida in FEW_SHOT:
        messages.append({"role": "user", "content": entrada})
        messages.append({"role": "assistant", "content": json.dumps(saida, ensure_ascii=False)})
    messages.append({"role": "user", "content": f"{alerta}\n{contexto}"})
    return messages

## Passo 8 — Rodar o baseline na amostra

**O que fazer:** carregar os 20 alertas da amostra de baseline e gerar a triagem de cada um com o prompt do Passo 7.

**Por que:** é a medição real do "prompt sozinho resolve?". A geração usa `do_sample=False` (decodificação gulosa) para que o resultado seja reprodutível: rodar de novo dá a mesma saída.

**Solução:**

> `split="train"` é necessário porque, ao carregar um arquivo avulso com `data_files`, a biblioteca `datasets` coloca tudo num split chamado `train`. O nome é só convenção e não quer dizer que esses dados serão usados para treinar.

In [ ]:
from datasets import load_dataset

DATASET_ID = "org-da-turma/alertas-soc"

amostra = load_dataset(DATASET_ID, data_files="amostra_baseline.jsonl", split="train")
print(len(amostra))  # 20
print(amostra[0])


def gerar(messages: list[dict]) -> str:
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(inputs, max_new_tokens=200, do_sample=False)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)


saidas = [gerar(montar_mensagens(ex["alerta"], ex["contexto"])) for ex in amostra]
print(saidas[0])

## Passo 9 — Medir o resultado do baseline

**O que fazer:** validar cada saída e calcular as métricas do Passo 4 sobre os 20 casos.

**Por que:** a avaliação precisa ser **estrita** e igual à que será usada na Fase 6, para que a comparação baseline × fine-tuned seja justa. Saída inválida conta como erro em todas as métricas: na prática, o sistema não consegue usá-la.

**Solução:**

In [ ]:
VEREDITOS = {"falso_positivo", "verdadeiro_positivo"}
SEVERIDADES = {"baixa", "media", "alta", "critica"}
CHAVES = {"veredito", "severidade", "justificativa"}


def validar(saida: str) -> dict | None:
    try:
        d = json.loads(saida)
    except json.JSONDecodeError:
        return None
    if not isinstance(d, dict) or set(d) != CHAVES:
        return None
    if d["veredito"] not in VEREDITOS or d["severidade"] not in SEVERIDADES:
        return None
    return d


previstos = [validar(s) for s in saidas]
reais = [ex["triagem"] for ex in amostra]
n = len(reais)

json_valido = sum(p is not None for p in previstos) / n
acc_veredito = sum(p is not None and p["veredito"] == r["veredito"] for p, r in zip(previstos, reais)) / n
acc_severidade = sum(p is not None and p["severidade"] == r["severidade"] for p, r in zip(previstos, reais)) / n

criticos = [(p, r) for p, r in zip(previstos, reais) if r["severidade"] == "critica"]
recall_critica = sum(p is not None and p["severidade"] == "critica" for p, r in criticos) / len(criticos)

print(f"JSON válido:         {json_valido:.0%}")
print(f"Acurácia veredito:   {acc_veredito:.0%}")
print(f"Acurácia severidade: {acc_severidade:.0%}")
print(f"Recall de crítica:   {recall_critica:.0%} ({len(criticos)} críticos na amostra)")

Também vale inspecionar as falhas, porque elas orientam as próximas fases:

In [ ]:
for s, p in zip(saidas, previstos):
    if p is None:
        print("INVÁLIDA:", s[:150])

> **Falha típica:** o modelo devolve o JSON dentro de um bloco ` ```json ... ``` ` ou acrescenta um texto antes ("Aqui está a triagem:"). Pelo critério estrito isso é inválido, e com razão: o sistema que consome a triagem quebraria. É um dos problemas que o fine-tuning resolve, ao ensinar o formato exato.

## Passo 10 — Interpretar o resultado e decidir

**O que fazer:** comparar o baseline com as metas do Passo 4 e decidir se o fine-tuning se justifica.

**Por que:** é a pergunta central da fase. Fine-tuning tem custo (dados, GPU, manutenção) e só vale se trouxer ganho que o prompt não traz.

**Solução:** regra de decisão:

| Situação | Decisão |
|---|---|
| Baseline atinge **todas** as metas | **Não fazer fine-tuning.** Usar o prompt em produção. Registrar isso no relatório |
| Baseline falha em formato (JSON) mas acerta a severidade | O fine-tuning se justifica pela **consistência de formato** |
| Baseline falha em severidade, principalmente em "crítica" | O fine-tuning se justifica por **aprender a política da organização** |
| Baseline falha em ambos | O fine-tuning se justifica pelos dois motivos |

Exemplo de registro preenchido. **Os valores são ilustrativos:** os números reais são os obtidos no Passo 9.

| Métrica | Meta | Baseline (few-shot, 0.5B) | Atingiu? |
|---|---|---|---|
| JSON válido | ≥ 98% | 70% (14/20) | Não |
| Acurácia de veredito | ≥ 85% | 65% (13/20) | Não |
| Recall de "crítica" | ≥ 90% | 40% (2/5) | Não |

**Conclusão (exemplo):** com o melhor prompt few-shot, o modelo 0.5B não atinge nenhuma meta. Ele erra o formato em 30% dos casos e deixa de marcar 3 de 5 alertas críticos, quase sempre porque ignora o contexto do ativo (trata o sinal forte sempre igual). O fine-tuning se justifica pelos dois motivos: consistência de formato e aplicação da política de severidade.

> Se o baseline atingir as metas, a conclusão correta é "não faria fine-tuning neste caso". Para fins didáticos o projeto continua, mas essa conclusão é válida se estiver bem justificada.

## Passo 11 — Ficha da Fase 1 (entregável)

**O que fazer:** consolidar os passos anteriores numa ficha de uma página.

**Por que:** a ficha é o "contrato" do projeto. As Fases 2 a 7 consultam esta ficha: os dados seguem a tarefa definida aqui, a avaliação usa estas metas e a Fase 6 compara com este baseline.

**Solução (modelo de ficha preenchida):**

```markdown
# Ficha da Fase 1

## Tarefa
Entrada: alerta com sinal técnico + contexto enriquecido.
Saída: JSON com veredito, severidade e justificativa.
Tipo: SFT de LLM pequeno instruction-tuned, saída em formato restrito.
Fora do escopo: enriquecimento, mapeamento MITRE, ação de resposta.

## Metas (definidas antes do treino)
- JSON válido ≥ 98%
- Acurácia de veredito ≥ 85%
- Recall de "crítica" ≥ 90% (métrica de negócio)
- Justificativa coerente ≥ 8/10
- Superar o baseline

## Modelo base
Qwen/Qwen2.5-0.5B-Instruct — Apache 2.0, sem gating
Licença: https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct
Motivo: licença aberta, sem bloqueio de acesso, bom em português,
treina numa T4 no tempo do projeto, barato para rodar on-premises.
Memória na T4 (fp16): ~1 GB.

## Baseline (few-shot, 20 alertas)
JSON válido: __% | Acurácia de veredito: __% | Recall de crítica: __%
Principais falhas observadas: ______

## Decisão
O fine-tuning [se justifica / não se justifica] porque ______.
```

### Checklist de saída da Fase 1

- [ ] Ambiente com GPU T4 e token apenas nos *Secrets* (Passo 1).
- [ ] Tarefa definida com entrada, saída e fora do escopo (Passo 2).
- [ ] Exercício da política resolvido (Passo 3).
- [ ] Metas fixadas e justificadas antes do treino (Passo 4).
- [ ] Modelo escolhido, com licença conferida no Hub (Passos 5 e 6).
- [ ] Baseline executado e medido nos 20 alertas (Passos 7 a 9).
- [ ] Decisão sobre o fine-tuning registrada (Passo 10).
- [ ] Ficha da Fase 1 preenchida (Passo 11).